# Phase 2 — Collaborative Filtering: Evaluation & Analysis
Make sure `preprocess.py` has been run before this notebook.

In [ ]:
import sys
sys.path.append('../../')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.models.collaborative import CollaborativeFilter

## 1. Train the model (or load if already saved)

In [ ]:
cf = CollaborativeFilter()

# Use sample_frac=0.1 for a quick test run; set to 1.0 for full training
metrics = cf.train(
    n_factors   = 100,
    n_epochs    = 30,
    lr_all      = 0.005,
    reg_all     = 0.02,
    sample_frac = 0.1,   # ← increase to 1.0 for production
)
print(metrics)

## 2. Top-N recommendations for a sample user

In [ ]:
# Pick any userId present in the ratings file
ratings = pd.read_csv('../../data/processed/ratings_clean.csv')
sample_user = ratings['userId'].iloc[0]
print(f'Generating recs for user {sample_user}')

recs = cf.top_n(user_id=sample_user, n=10)
recs

## 3. Predicted vs actual ratings — spot check

In [ ]:
# Get 50 movies this user actually rated and compare predictions
user_ratings = ratings[ratings['userId'] == sample_user].head(50)
user_ratings = user_ratings.copy()
user_ratings['predicted'] = user_ratings['movieId'].apply(
    lambda mid: cf.predict_rating(sample_user, mid)
)

plt.figure(figsize=(8, 5))
plt.scatter(user_ratings['rating'], user_ratings['predicted'], alpha=0.6)
plt.plot([0.5, 5], [0.5, 5], 'r--', label='Perfect prediction')
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title(f'Actual vs Predicted — User {sample_user}')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Rating prediction distribution across all movies

In [ ]:
recs_all = cf.top_n(user_id=sample_user, n=50)
recs_all['predicted_rating'].hist(bins=20, color='steelblue', edgecolor='white')
plt.title('Distribution of Top-50 Predicted Ratings')
plt.xlabel('Predicted Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 5. MLflow — view tracked runs

In [ ]:
import mlflow
# Run `mlflow ui` in terminal to see the full dashboard
runs = mlflow.search_runs(experiment_names=['movie-recommender-svd'])
runs[['run_id', 'params.n_factors', 'params.n_epochs',
      'metrics.rmse', 'metrics.mae', 'start_time']].head(10)